# Inspect estimated timetable XML keys

This cell lists the XML element names (tags), their occurrence counts, and any attribute names in the compressed SIRI file.

In [1]:
import gzip
from collections import Counter
from pathlib import Path
import xml.etree.ElementTree as ET

xml_file = Path(
    "../data/estimated_timetable/date=2026-10-24/"
    "estimated_timetable_20261024T000015Z.xml.gz"
)

def local_name(name):
    """Remove an XML namespace from a tag or attribute name."""
    return name.rsplit("}", 1)[-1]

with gzip.open(xml_file, "rb") as compressed_xml:
    root = ET.parse(compressed_xml).getroot()

tag_counts = Counter(local_name(element.tag) for element in root.iter())
attribute_counts = Counter(
    local_name(attribute)
    for element in root.iter()
    for attribute in element.attrib
)

print(f"XML file: {xml_file}")
print("\nElement keys (tag: count):")
for tag, count in sorted(tag_counts.items()):
    print(f"{tag}: {count}")

print("\nAttribute keys (attribute: count):")
for attribute, count in sorted(attribute_counts.items()):
    print(f"{attribute}: {count}")

XML file: ..\data\estimated_timetable\date=2026-10-24\estimated_timetable_20261024T000015Z.xml.gz

Element keys (tag: count):
ActualArrivalTime: 908
ActualDepartureTime: 936
AimedArrivalTime: 2969
AimedDepartureTime: 2969
ArrivalBoardingActivity: 294
ArrivalStatus: 1991
Cancellation: 347
DataFrameRef: 166
DataSource: 166
DatedVehicleJourneyRef: 166
DepartureBoardingActivity: 260
DepartureStatus: 1927
DestinationDisplay: 1991
DirectionRef: 166
EstimatedCall: 2190
EstimatedCalls: 159
EstimatedJourneyVersionFrame: 1
EstimatedTimetableDelivery: 1
EstimatedVehicleJourney: 166
ExpectedArrivalTime: 2065
ExpectedDepartureTime: 2039
FramedVehicleJourneyRef: 166
IsCompleteStopSequence: 166
JourneyNote: 96
LineRef: 166
Monitored: 166
OperatorRef: 166
Order: 3135
OriginName: 69
PredictionInaccurate: 209
ProducerRef: 1
RecordedAtTime: 167
RecordedCall: 945
RecordedCalls: 35
RequestStop: 41
ResponseTimestamp: 2
ServiceDelivery: 1
Siri: 1
StopPointName: 2926
StopPointRef: 3135
VehicleMode: 165

Attri

In [3]:
from collections import defaultdict

journeys_by_vehicle_type = defaultdict(list)

for journey in root.iter():
    if local_name(journey.tag) != "EstimatedVehicleJourney":
        continue

    vehicle_mode = next(
        (child.text.strip() for child in journey
         if local_name(child.tag) == "VehicleMode" and child.text),
        "Unknown",
    )
    journeys_by_vehicle_type[vehicle_mode].append(journey)

for vehicle_type, journeys in sorted(journeys_by_vehicle_type.items()):
    print(f"{vehicle_type}: {len(journeys)} journeys")

Unknown: 1 journeys
bus: 68 journeys
ferry: 97 journeys


In [3]:
arrival_times = [
    element.text.strip()
    for element in root.iter()
    if local_name(element.tag) == "ActualArrivalTime"
    and element.text
    and element.text.strip()
]

print("Arrival-time elements with values:", len(arrival_times))
print("Unique arrival timestamps:", len(set(arrival_times)))

print("\nFirst 10 arrival timestamps:")
for timestamp in arrival_times[:10]:
    print(timestamp)

Arrival-time elements with values: 908
Unique arrival timestamps: 896

First 10 arrival timestamps:
2026-10-24T01:40:39.336+02:00
2026-10-24T01:41:04.325+02:00
2026-10-24T01:41:25.345+02:00
2026-10-24T01:41:46.343+02:00
2026-10-24T01:42:29.342+02:00
2026-10-24T01:43:36.339+02:00
2026-10-24T01:44:16.345+02:00
2026-10-24T01:44:48.338+02:00
2026-10-24T01:45:20.345+02:00
2026-10-24T01:45:55.353+02:00


In [4]:

# Find all estimated vehicle journeys
journeys = [
    element
    for element in root.iter()
    if local_name(element.tag) == "EstimatedVehicleJourney"
]

print("Total journeys:", len(journeys))

if journeys:
    # Select the first journey
    journey = journeys[0]

    print("\nFull XML data for one journey:\n")
    print(ET.tostring(journey, encoding="unicode"))

else:
    print("No EstimatedVehicleJourney elements found.")


Total journeys: 166

Full XML data for one journey:

<ns0:EstimatedVehicleJourney xmlns:ns0="http://www.siri.org.uk/siri"><ns0:RecordedAtTime>2026-10-24T00:01:54.544586848+02:00</ns0:RecordedAtTime><ns0:LineRef>KOL:Line:30_1130</ns0:LineRef><ns0:DirectionRef>inbound</ns0:DirectionRef><ns0:FramedVehicleJourneyRef><ns0:DataFrameRef>2026-10-24</ns0:DataFrameRef><ns0:DatedVehicleJourneyRef>KOL:ServiceJourney:1130_250903083242106_4040</ns0:DatedVehicleJourneyRef></ns0:FramedVehicleJourneyRef><ns0:Cancellation>false</ns0:Cancellation><ns0:VehicleMode>ferry</ns0:VehicleMode><ns0:OperatorRef>KOL:Operator:596</ns0:OperatorRef><ns0:JourneyNote xml:lang="NO">C-ruta</ns0:JourneyNote><ns0:Monitored>true</ns0:Monitored><ns0:DataSource>KOL</ns0:DataSource><ns0:EstimatedCalls><ns0:EstimatedCall><ns0:StopPointRef>NSR:Quay:99269</ns0:StopPointRef><ns0:Order>1</ns0:Order><ns0:Cancellation>false</ns0:Cancellation><ns0:PredictionInaccurate>false</ns0:PredictionInaccurate><ns0:ArrivalBoardingActivity>noAlig

In [4]:

from collections import Counter

journeys = [
    element
    for element in root.iter()
    if local_name(element.tag) == "EstimatedVehicleJourney"
]

vehicle_ids = []
missing_vehicle_ref = 0

for journey in journeys:
    vehicle_ref = next(
        (
            element.text.strip()
            for element in journey.iter()
            if local_name(element.tag) == "VehicleRef"
            and element.text
            and element.text.strip()
        ),
        None,
    )

    if vehicle_ref:
        vehicle_ids.append(vehicle_ref)
    else:
        missing_vehicle_ref += 1

unique_vehicles = set(vehicle_ids)

print("Journey records:", len(journeys))
print("Vehicle references found:", len(vehicle_ids))
print("Unique vehicle references:", len(unique_vehicles))
print("Journeys without VehicleRef:", missing_vehicle_ref)
print("Vehicle references appearing multiple times:")

for vehicle_id, count in Counter(vehicle_ids).items():
    if count > 1:
        print(f"  {vehicle_id}: {count} journeys")


Journey records: 166
Vehicle references found: 0
Unique vehicle references: 0
Journeys without VehicleRef: 166
Vehicle references appearing multiple times:


## Explore the timetable with PySpark

These cells turn the XML journeys and estimated calls into a Spark DataFrame, then summarize route activity, stop activity, and arrival-time differences. Run the XML-loading cell above first. PySpark also needs a compatible Java runtime; for Spark 4.x, install Java 17 or newer and set `JAVA_HOME` before starting Jupyter.

In [5]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, IntegerType, StringType, StructField, StructType

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("EnturEstimatedTimetableAnalysis")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

def child_text(element, name):
    """Return the first non-empty direct child value with this local tag name."""
    for child in element:
        if local_name(child.tag) == name and child.text and child.text.strip():
            return child.text.strip()
    return None

def descendant_text(element, name):
    """Return the first non-empty value below an element, including nested children."""
    for child in element.iter():
        if local_name(child.tag) == name and child.text and child.text.strip():
            return child.text.strip()
    return None

def as_bool(value):
    if value is None:
        return None
    return value.lower() == "true"

schema = StructType([
    StructField("journey_id", StringType(), True),
    StructField("line_ref", StringType(), True),
    StructField("route_id", StringType(), True),
    StructField("vehicle_mode", StringType(), True),
    StructField("direction", StringType(), True),
    StructField("journey_note", StringType(), True),
    StructField("recorded_at", StringType(), True),
    StructField("stop_ref", StringType(), True),
    StructField("stop_order", IntegerType(), True),
    StructField("aimed_arrival", StringType(), True),
    StructField("expected_arrival", StringType(), True),
    StructField("actual_arrival", StringType(), True),
    StructField("aimed_departure", StringType(), True),
    StructField("expected_departure", StringType(), True),
    StructField("actual_departure", StringType(), True),
    StructField("journey_cancelled", BooleanType(), True),
    StructField("stop_cancelled", BooleanType(), True),
    StructField("prediction_inaccurate", BooleanType(), True),
    StructField("arrival_boarding", StringType(), True),
    StructField("departure_boarding", StringType(), True),
])

rows = []
for journey in root.iter():
    if local_name(journey.tag) != "EstimatedVehicleJourney":
        continue

    line_ref = child_text(journey, "LineRef")
    line_code = line_ref.rsplit(":", 1)[-1] if line_ref else None
    route_id = line_code.split("_", 1)[0] if line_code else None

    for call in journey.iter():
        if local_name(call.tag) != "EstimatedCall":
            continue

        order = child_text(call, "Order")
        rows.append((
            descendant_text(journey, "DatedVehicleJourneyRef"),
            line_ref,
            route_id,
            child_text(journey, "VehicleMode"),
            child_text(journey, "DirectionRef"),
            child_text(journey, "JourneyNote"),
            child_text(journey, "RecordedAtTime"),
            child_text(call, "StopPointRef"),
            int(order) if order and order.isdigit() else None,
            child_text(call, "AimedArrivalTime"),
            child_text(call, "ExpectedArrivalTime"),
            child_text(call, "ActualArrivalTime"),
            child_text(call, "AimedDepartureTime"),
            child_text(call, "ExpectedDepartureTime"),
            child_text(call, "ActualDepartureTime"),
            as_bool(child_text(journey, "Cancellation")),
            as_bool(child_text(call, "Cancellation")),
            as_bool(child_text(call, "PredictionInaccurate")),
            child_text(call, "ArrivalBoardingActivity"),
            child_text(call, "DepartureBoardingActivity"),
        ))

timetable_df = spark.createDataFrame(rows, schema=schema)
print(f"Spark rows (one row per estimated stop call): {timetable_df.count():,}")
timetable_df.printSchema()
timetable_df.show(10, truncate=False)

Spark rows (one row per estimated stop call): 2,190
root
 |-- journey_id: string (nullable = true)
 |-- line_ref: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- vehicle_mode: string (nullable = true)
 |-- direction: string (nullable = true)
 |-- journey_note: string (nullable = true)
 |-- recorded_at: string (nullable = true)
 |-- stop_ref: string (nullable = true)
 |-- stop_order: integer (nullable = true)
 |-- aimed_arrival: string (nullable = true)
 |-- expected_arrival: string (nullable = true)
 |-- actual_arrival: string (nullable = true)
 |-- aimed_departure: string (nullable = true)
 |-- expected_departure: string (nullable = true)
 |-- actual_departure: string (nullable = true)
 |-- journey_cancelled: boolean (nullable = true)
 |-- stop_cancelled: boolean (nullable = true)
 |-- prediction_inaccurate: boolean (nullable = true)
 |-- arrival_boarding: string (nullable = true)
 |-- departure_boarding: string (nullable = true)

+-------------------------------

### 1. Snapshot overview

The source file is a single snapshot, not a full-day history. These counts describe the journeys and stop calls present in that snapshot.

In [6]:
timetable_df.agg(
    F.count("*").alias("stop_calls"),
    F.countDistinct("journey_id").alias("journeys"),
    F.countDistinct("line_ref").alias("distinct_line_refs"),
    F.countDistinct("stop_ref").alias("distinct_stops"),
    F.sum(F.col("journey_cancelled").cast("int")).alias("cancelled_journey_calls"),
    F.sum(F.col("stop_cancelled").cast("int")).alias("cancelled_stop_calls"),
).show(truncate=False)

timetable_df.groupBy("vehicle_mode").agg(
    F.countDistinct("journey_id").alias("journeys"),
    F.count("*").alias("stop_calls"),
).orderBy(F.desc("journeys")).show(truncate=False)

+----------+--------+------------------+--------------+-----------------------+--------------------+
|stop_calls|journeys|distinct_line_refs|distinct_stops|cancelled_journey_calls|cancelled_stop_calls|
+----------+--------+------------------+--------------+-----------------------+--------------------+
|2190      |159     |22                |984           |0                      |44                  |
+----------+--------+------------------+--------------+-----------------------+--------------------+

+------------+--------+----------+
|vehicle_mode|journeys|stop_calls|
+------------+--------+----------+
|ferry       |95      |199       |
|bus         |63      |1989      |
|NULL        |1       |2         |
+------------+--------+----------+



### 2. Compare route activity

This ranks route codes by distinct journeys and stop calls. `line_ref` is the full Entur identifier; `route_id` is the leading code extracted from it.

In [7]:
route_summary = (
    timetable_df.groupBy("route_id", "line_ref", "vehicle_mode")
    .agg(
        F.countDistinct("journey_id").alias("journeys"),
        F.count("*").alias("stop_calls"),
        F.sum(F.col("journey_cancelled").cast("int")).alias("cancelled_calls"),
    )
    .orderBy(F.desc("journeys"), "route_id")
)
route_summary.show(30, truncate=False)

+--------+----------------+------------+--------+----------+---------------+
|route_id|line_ref        |vehicle_mode|journeys|stop_calls|cancelled_calls|
+--------+----------------+------------+--------+----------+---------------+
|26      |KOL:Line:26_5958|ferry       |58      |117       |0              |
|30      |KOL:Line:30_1130|ferry       |34      |68        |0              |
|8       |KOL:Line:8_1305 |bus         |8       |415       |NULL           |
|8       |KOL:Line:8_1001 |bus         |6       |160       |NULL           |
|8       |KOL:Line:8_1307 |bus         |6       |179       |NULL           |
|8       |KOL:Line:8_1306 |bus         |6       |188       |NULL           |
|8       |KOL:Line:8_1006 |bus         |5       |295       |NULL           |
|8       |KOL:Line:8_1104 |bus         |5       |102       |NULL           |
|8       |KOL:Line:8_1109 |bus         |5       |91        |NULL           |
|8       |KOL:Line:8_1310 |bus         |3       |69        |NULL           |

### 3. Most frequently served stops

Stop references are used because this feed does not consistently include human-readable stop names inside each estimated call.

In [8]:
timetable_df.groupBy("stop_ref").agg(
    F.count("*").alias("estimated_calls"),
    F.countDistinct("journey_id").alias("distinct_journeys"),
    F.countDistinct("route_id").alias("routes_serving_stop"),
    F.sum(F.col("stop_cancelled").cast("int")).alias("cancelled_calls"),
).orderBy(F.desc("estimated_calls")).show(20, truncate=False)

+---------------+---------------+-----------------+-------------------+---------------+
|stop_ref       |estimated_calls|distinct_journeys|routes_serving_stop|cancelled_calls|
+---------------+---------------+-----------------+-------------------+---------------+
|NSR:Quay:45757 |58             |58               |1                  |0              |
|NSR:Quay:47191 |53             |53               |1                  |0              |
|NSR:Quay:99704 |34             |34               |1                  |0              |
|NSR:Quay:99269 |34             |34               |1                  |0              |
|NSR:Quay:46964 |17             |17               |1                  |1              |
|NSR:Quay:46957 |13             |13               |1                  |NULL           |
|NSR:Quay:46970 |9              |9                |1                  |NULL           |
|NSR:Quay:103468|9              |9                |1                  |NULL           |
|NSR:Quay:47945 |8              

### 4. Expected arrival versus aimed arrival

This measures the prediction's deviation from the scheduled arrival: positive means the expected time is later than aimed, negative means earlier. It is not a measured actual delay. The calculation uses whole seconds and skips missing timestamps and canceled calls.

In [9]:
arrival_predictions = (
    timetable_df
    .filter(
        F.col("aimed_arrival").isNotNull()
        & F.col("expected_arrival").isNotNull()
        & (F.col("stop_cancelled").isNull() | ~F.col("stop_cancelled"))
        & (F.col("journey_cancelled").isNull() | ~F.col("journey_cancelled"))
    )
    .withColumn("aimed_ts", F.to_timestamp("aimed_arrival"))
    .withColumn("expected_ts", F.to_timestamp("expected_arrival"))
    .withColumn(
        "prediction_delta_seconds",
        F.unix_timestamp("expected_ts") - F.unix_timestamp("aimed_ts"),
    )
    .filter(F.col("aimed_ts").isNotNull() & F.col("expected_ts").isNotNull())
)

arrival_predictions.select(
    "route_id", "stop_ref", "aimed_arrival", "expected_arrival",
    "prediction_delta_seconds",
).show(20, truncate=False)

arrival_predictions.agg(
    F.count("*").alias("compared_arrivals"),
    F.round(F.avg("prediction_delta_seconds"), 1).alias("mean_delta_seconds"),
    F.expr("percentile_approx(prediction_delta_seconds, 0.5)").alias("median_delta_seconds"),
    F.sum((F.col("prediction_delta_seconds") > 300).cast("int")).alias("over_5m_late"),
    F.sum((F.col("prediction_delta_seconds") < -300).cast("int")).alias("over_5m_early"),
).show(truncate=False)

arrival_predictions.groupBy("route_id").agg(
    F.count("*").alias("arrival_predictions"),
    F.round(F.avg("prediction_delta_seconds") / 60, 2).alias("mean_deviation_minutes"),
    F.expr("percentile_approx(prediction_delta_seconds, 0.5)").alias("median_deviation_seconds"),
    F.sum((F.col("prediction_delta_seconds") > 300).cast("int")).alias("over_5m_late"),
).orderBy(F.desc("arrival_predictions")).show(30, truncate=False)

+--------+--------------+-------------------------+-----------------------------+------------------------+
|route_id|stop_ref      |aimed_arrival            |expected_arrival             |prediction_delta_seconds|
+--------+--------------+-------------------------+-----------------------------+------------------------+
|30      |NSR:Quay:99704|2026-10-24T18:54:00+02:00|2026-10-24T18:54:00+02:00    |0                       |
|26      |NSR:Quay:45757|2026-10-24T16:53:00+02:00|2026-10-24T16:53:00+02:00    |0                       |
|8       |NSR:Quay:47081|2026-10-24T03:05:00+02:00|2026-10-24T03:05:00+02:00    |0                       |
|8       |NSR:Quay:45407|2026-10-24T03:10:00+02:00|2026-10-24T03:10:00+02:00    |0                       |
|8       |NSR:Quay:46631|2026-10-24T03:12:00+02:00|2026-10-24T03:12:00+02:00    |0                       |
|8       |NSR:Quay:46858|2026-10-24T03:18:00+02:00|2026-10-24T03:18:00+02:00    |0                       |
|8       |NSR:Quay:45536|2026-10-24T0

### 5. Data completeness and observed arrivals

Check how much of this snapshot has arrival forecasts, actual arrival timestamps, and prediction-quality flags. Actual-versus-aimed is only computed where both values are present.

In [10]:
timetable_df.agg(
    F.count("*").alias("stop_calls"),
    F.sum(F.col("aimed_arrival").isNotNull().cast("int")).alias("with_aimed_arrival"),
    F.sum(F.col("expected_arrival").isNotNull().cast("int")).alias("with_expected_arrival"),
    F.sum(F.col("actual_arrival").isNotNull().cast("int")).alias("with_actual_arrival"),
    F.sum(F.col("prediction_inaccurate").cast("int")).alias("flagged_prediction_inaccurate"),
    F.sum(F.col("journey_id").isNull().cast("int")).alias("missing_journey_id"),
    F.sum(F.col("stop_ref").isNull().cast("int")).alias("missing_stop_ref"),
).show(truncate=False)

actual_arrivals = (
    timetable_df
    .filter(
        F.col("aimed_arrival").isNotNull()
        & F.col("actual_arrival").isNotNull()
        & (F.col("stop_cancelled").isNull() | ~F.col("stop_cancelled"))
        & (F.col("journey_cancelled").isNull() | ~F.col("journey_cancelled"))
    )
    .withColumn("aimed_ts", F.to_timestamp("aimed_arrival"))
    .withColumn("actual_ts", F.to_timestamp("actual_arrival"))
    .withColumn(
        "actual_delta_seconds",
        F.unix_timestamp("actual_ts") - F.unix_timestamp("aimed_ts"),
    )
    .filter(F.col("aimed_ts").isNotNull() & F.col("actual_ts").isNotNull())
)

actual_arrivals.agg(
    F.count("*").alias("actual_arrivals_compared"),
    F.round(F.avg("actual_delta_seconds"), 1).alias("mean_actual_delta_seconds"),
    F.expr("percentile_approx(actual_delta_seconds, 0.5)").alias("median_actual_delta_seconds"),
    F.sum((F.col("actual_delta_seconds") > 300).cast("int")).alias("over_5m_late"),
    F.sum((F.col("actual_delta_seconds") < -300).cast("int")).alias("over_5m_early"),
).show(truncate=False)

actual_arrivals.groupBy("route_id").agg(
    F.count("*").alias("actual_arrivals"),
    F.round(F.avg("actual_delta_seconds") / 60, 2).alias("mean_actual_delta_minutes"),
    F.expr("percentile_approx(actual_delta_seconds, 0.5)").alias("median_actual_delta_seconds"),
).orderBy(F.desc("actual_arrivals")).show(30, truncate=False)

+----------+------------------+---------------------+-------------------+-----------------------------+------------------+----------------+
|stop_calls|with_aimed_arrival|with_expected_arrival|with_actual_arrival|flagged_prediction_inaccurate|missing_journey_id|missing_stop_ref|
+----------+------------------+---------------------+-------------------+-----------------------------+------------------+----------------+
|2190      |2059              |2059                 |0                  |0                            |0                 |0               |
+----------+------------------+---------------------+-------------------+-----------------------------+------------------+----------------+

+------------------------+-------------------------+---------------------------+------------+-------------+
|actual_arrivals_compared|mean_actual_delta_seconds|median_actual_delta_seconds|over_5m_late|over_5m_early|
+------------------------+-------------------------+---------------------------+---

When finished, release the local Spark session with `spark.stop()`.